# Proyecto: Clasificación de Subgéneros de Jazz (FMA)

## 1. Configuración del Entorno y Almacenamiento

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models
from imblearn.over_sampling import SMOTE
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

print("Librerías cargadas correctamente.")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Rutas relativas (asumiendo que el notebook está en /notebooks y los datos en /data)
DATA_DIR = "../data/"

print("Cargando matrices preprocesadas...")
X_tabular = np.load(f"{DATA_DIR}X_tabular.npy")
X_cnn = np.load(f"{DATA_DIR}X_cnn.npy")
y_encoded = np.load(f"{DATA_DIR}y_encoded.npy")
classes = np.load(f"{DATA_DIR}classes.npy")

print(f"Forma de X_tabular (Random Forest): {X_tabular.shape}")
print(f"Forma de X_cnn (Red Neuronal): {X_cnn.shape}")
print(f"Clases detectadas: {classes}")

# Train/Test Split común para ambos
X_train_t, X_test_t, y_train, y_test = train_test_split(X_tabular, y_encoded, test_size=0.2, random_state=42)
X_train_m, X_test_m, _, _ = train_test_split(X_cnn, y_encoded, test_size=0.2, random_state=42)
todas_las_clases = np.arange(len(classes))

In [ ]:
print("--- Balanceando datos con SMOTE ---")
smote = SMOTE(random_state=42)
X_train_t_smote, y_train_smote = smote.fit_resample(X_train_t, y_train)

print("Entrenando Random Forest...")
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_t_smote, y_train_smote)
y_pred_rf = rf_model.predict(X_test_t)

print("\n=== REPORTE RANDOM FOREST (CON SMOTE) ===")
print(classification_report(y_test, y_pred_rf, labels=todas_las_clases, target_names=classes, zero_division=0))

In [ ]:
print("--- Configurando CNN con Data Augmentation ---")
# 1. Pesos suavizados (Clipping)
pesos_crudos = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
pesos_suavizados = np.clip(pesos_crudos, a_min=1.0, a_max=4.0)
pesos_dict = dict(enumerate(pesos_suavizados))

# 2. Generador de imágenes alteradas
datagen = ImageDataGenerator(width_shift_range=0.1, height_shift_range=0.1, fill_mode='nearest')

# 3. Construir modelo
cnn_model = models.Sequential([
    layers.Input(shape=(128, 128, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(classes), activation='softmax')
])

cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Entrenar
history = cnn_model.fit(
    datagen.flow(X_train_m, y_train, batch_size=32),
    epochs=25, 
    class_weight=pesos_dict,
    validation_data=(X_test_m, y_test),
    verbose=1
)

y_pred_cnn = np.argmax(cnn_model.predict(X_test_m), axis=1)
print("\n=== REPORTE CNN (CON DATA AUGMENTATION) ===")
print(classification_report(y_test, y_pred_cnn, labels=todas_las_clases, target_names=classes, zero_division=0))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(confusion_matrix(y_test, y_pred_rf, labels=todas_las_clases), annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes, ax=ax[0])
ax[0].set_title('Matriz de Confusión: Random Forest (SMOTE)')
ax[0].tick_params(axis='x', rotation=45)

sns.heatmap(confusion_matrix(y_test, y_pred_cnn, labels=todas_las_clases), annot=True, fmt='d', cmap='Oranges', 
            xticklabels=classes, yticklabels=classes, ax=ax[1])
ax[1].set_title('Matriz de Confusión: CNN (Data Augmentation)')
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()